In [ ]:
!pip install -U sarvamai

  Using cached sarvamai-0.1.28-py3-none-any.whl.metadata (26 kB)
Using cached sarvamai-0.1.28-py3-none-any.whl (269 kB)


In [ ]:
import os
from google.colab import userdata
from sarvamai import SarvamAI
from sarvamai.core.api_error import ApiError

# =====================================================================
# CONFIGURATION: Set your uploaded file name here
# =====================================================================
YOUR_AUDIO_FILE = "/content/AUD-20260402-WA0018.m4a"

# =====================================================================
# PIPELINE EXECUTION
# =====================================================================
print("🚀 Executing Task 1 (Dual Punjabi Output Mode)...")

if not os.path.exists(YOUR_AUDIO_FILE):
    print(f"❌ File Not Found: Cannot find '{YOUR_AUDIO_FILE}'")
    print("👉 Click the Folder icon on the left sidebar and upload your audio file directly there.")
else:
    # Pull the API token securely from Colab Secrets
    try:
        API_KEY = userdata.get('SARVAM_API_KEY')
        client = SarvamAI(api_subscription_key=API_KEY)
    except Exception:
        print("❌ Secret Key Error: 'SARVAM_API_KEY' not found in Colab Secrets (🔑 icon).")
        client = None

    if client:
        print(f"🎵 Processing File: {os.path.basename(YOUR_AUDIO_FILE)}")

        try:
            # --- FILE 1: Standard Punjabi Transcription ---
            print("🔄 Generating Standard Punjabi Transcript...")
            with open(YOUR_AUDIO_FILE, "rb") as audio_file:
                stt_response_1 = client.speech_to_text.transcribe(
                    file=audio_file,
                    model="saaras:v3",
                    language_code="pa-IN", # Correct BCP-47 identifier for Punjabi
                    mode="transcribe"      # Standard clean text formatting
                )
            punjabi_standard = stt_response_1.transcript

            # --- FILE 2: Verbatim Punjabi Transcription ---
            print("🔄 Generating Verbatim Punjabi Transcript...")
            with open(YOUR_AUDIO_FILE, "rb") as audio_file:
                stt_response_2 = client.speech_to_text.transcribe(
                    file=audio_file,
                    model="saaras:v3",
                    language_code="pa-IN",
                    mode="verbatim"        # Preserves spoken filler words and exact raw phrasing
                )
            punjabi_verbatim = stt_response_2.transcript

            # --- STEP 3: Write Both Output Files ---
            out_path_standard = "/content/transcript_punjabi.txt"
            out_path_verbatim = "/content/transcript_punjabi_verbatim.txt"

            with open(out_path_standard, "w", encoding="utf-8") as f:
                f.write(punjabi_standard)

            with open(out_path_verbatim, "w", encoding="utf-8") as f:
                f.write(punjabi_verbatim)

            print("\n✨ TASK 1 COMPLETION SUCCESSFUL!")
            print(f"💾 File 1 Saved: {out_path_standard}")
            print(f"💾 File 2 Saved: {out_path_verbatim}")
            print("👉 Click the Refresh button in the file tree to view and download your text files!")

        except ApiError as e:
            print(f"\n❌ Sarvam API Error ({e.status_code}): {e.body}")
        except Exception as e:
            print(f"\n❌ Python Runtime Crash: {str(e)}")

🚀 Executing Task 1 (Dual Punjabi Output Mode)...
🎵 Processing File: AUD-20260402-WA0018.m4a
🔄 Generating Standard Punjabi Transcript...

❌ Sarvam API Error (400): {'error': {'message': 'Audio duration exceeds the maximum limit of 30 seconds. Please use the batch API for longer audio files.', 'code': 'invalid_request_error', 'request_id': '20260611_23e89db4-c9e9-4fe4-9bba-c6d6193d0a89'}}


In [ ]:
import os
import glob
import tempfile
import time
from google.colab import userdata
from sarvamai import SarvamAI
from sarvamai.core.api_error import ApiError

# =====================================================================
# CONFIGURATION: Set your long audio file name here
# =====================================================================
YOUR_AUDIO_FILE = "/content/AUD-20260402-WA0018.m4a"

# =====================================================================
# ASYNC BATCH PIPELINE ENGINE
# =====================================================================
print("🚀 Starting Task 1 (Batch Processing Mode)...")

if not os.path.exists(YOUR_AUDIO_FILE):
    print(f"❌ File Not Found: Cannot find '{YOUR_AUDIO_FILE}'")
    print("👉 Upload your long audio file directly to the left panel and check the filename.")
else:
    # Pull the API token securely from Colab Secrets
    try:
        API_KEY = userdata.get('SARVAM_API_KEY')
        client = SarvamAI(api_subscription_key=API_KEY)
    except Exception:
        print("❌ Secret Key Error: 'SARVAM_API_KEY' not found in Colab Secrets (🔑 icon).")
        client = None

    if client:
        print(f"📦 Initializing Batch Job for long file: {os.path.basename(YOUR_AUDIO_FILE)}")

        try:
            # --- STEP 1: Create a Batch Transcription Job ---
            # Using saaras:v3 and passing Punjabi ('pa-IN') to the job parameters
            job = client.speech_to_text_job.create_job(
                model="saaras:v3",
                mode="transcribe",
                language_code="pa-IN",
                with_diarization=False
            )

            # --- STEP 2: Upload long file and dispatch job execution ---
            print("📤 Uploading file chunks to Sarvam cloud storage...")
            job.upload_files(file_paths=[YOUR_AUDIO_FILE])

            print("⚙️ Processing audio script pipeline on remote server...")
            job.start()

            # --- STEP 3: Wait for compilation ---
            print("⏳ Waiting for batch job to complete (this may take a minute for long files)...")
            job.wait_until_complete()

            # --- STEP 4: Parse Transcript Result Payload ---
            # Download outputs to a temporary processing directory
            with tempfile.TemporaryDirectory() as temp_dir:
                job.download_outputs(output_dir=temp_dir)

                # Fetch generated JSON output results
                json_files = glob.glob(os.path.join(temp_dir, "*.json"))
                if not json_files:
                    raise Exception("Batch processing failed to yield output text logs.")

                # Read the successful payload text
                import json
                with open(json_files[0], 'r', encoding='utf-8') as f:
                    result_data = json.load(f)

                punjabi_text = result_data.get("transcript", "")

            print("✅ Native Punjabi Transcript Compiled successfully.")

            # --- STEP 5: Text Translation to Hindi Script ---
            print("🔄 Converting script layout to Hindi Devanagari...")
            translit_response = client.text.translate(
                input=punjabi_text,
                source_language_code="pa-IN",
                target_language_code="hi-IN",
                model="sarvam-translate:v1"
            )
            hindi_text = translit_response.translated_text
            print("   🔹 Hindi Text Extracted.")

            # --- STEP 6: Write Final Text Log Files ---
            punjabi_out = "/content/transcript_punjabi.txt"
            hindi_out = "/content/transcript_hindi.txt"

            with open(punjabi_out, "w", encoding="utf-8") as pf:
                pf.write(punjabi_text)

            with open(hindi_out, "w", encoding="utf-8") as hf:
                hf.write(hindi_text)

            print("\n✨ BATCH TASK 1 COMPLETION SUCCESSFUL!")
            print(f"💾 File 1 (Gurmukhi Script) Saved: {punjabi_out}")
            print(f"💾 File 2 (Devanagari Script) Saved: {hindi_out}")
            print("👉 Click the Refresh button on the sidebar to grab your files!")

        except ApiError as e:
            print(f"\n❌ Sarvam Batch API Error ({e.status_code}): {e.body}")
        except Exception as e:
            print(f"\n❌ Python Pipeline Crash: {str(e)}")

🚀 Starting Task 1 (Batch Processing Mode)...
📦 Initializing Batch Job for long file: AUD-20260402-WA0018.m4a
📤 Uploading file chunks to Sarvam cloud storage...
⚙️ Processing audio script pipeline on remote server...
⏳ Waiting for batch job to complete (this may take a minute for long files)...
✅ Native Punjabi Transcript Compiled successfully.
🔄 Converting script layout to Hindi Devanagari...

❌ Sarvam Batch API Error (400): {'error': {'message': 'body.input : String should have at most 2000 characters', 'code': 'invalid_request_error', 'request_id': '20260611_2bdd820e-4964-462e-850a-43b955e80ba2'}}


In [ ]:
import os
import glob
import tempfile
import time
import json
from google.colab import userdata
from sarvamai import SarvamAI
from sarvamai.core.api_error import ApiError

# =====================================================================
# CONFIGURATION: Set your long audio file name here
# =====================================================================
YOUR_AUDIO_FILE = "/content/AUD-20260402-WA0018.m4a"

# =====================================================================
# HELPER FUNCTION: Safely split text by sentences under character limit
# =====================================================================
def chunk_and_translate(client, text, max_chars=1500):
    """Splits text into safe chunks and translates them one by one"""
    # Split text into sentences using common Indian punctuation markers
    sentences = text.replace('।', '.').split('. ')
    chunks = []
    current_chunk = ""

    for sentence in sentences:
        if len(current_chunk) + len(sentence) + 2 < max_chars:
            current_chunk += sentence + "। "
        else:
            chunks.append(current_chunk.strip())
            current_chunk = sentence + "। "
    if current_chunk:
        chunks.append(current_chunk.strip())

    translated_chunks = []
    print(f"✂️ Text split into {len(chunks)} smaller segments for safe processing...")

    for idx, chunk in enumerate(chunks):
        if not chunk:
            continue
        print(f"   🔄 Processing segment {idx + 1}/{len(chunks)}...")
        try:
            translit_response = client.text.translate(
                input=chunk,
                source_language_code="pa-IN",
                target_language_code="hi-IN",
                model="sarvam-translate:v1"
            )
            translated_chunks.append(translit_response.translated_text)
            time.sleep(0.5) # Small pause to prevent API rate-limiting
        except ApiError as e:
            print(f"   ❌ Failed to translate segment {idx + 1}: {e.body}")

    return " ".join(translated_chunks)

# =====================================================================
# ASYNC BATCH PIPELINE ENGINE
# =====================================================================
print("🚀 Starting Task 1 (Chunked Batch Mode)...")

if not os.path.exists(YOUR_AUDIO_FILE):
    print(f"❌ File Not Found: Cannot find '{YOUR_AUDIO_FILE}'")
else:
    try:
        API_KEY = userdata.get('SARVAM_API_KEY')
        client = SarvamAI(api_subscription_key=API_KEY)
    except Exception:
        print("❌ Secret Key Error: 'SARVAM_API_KEY' not found in Colab Secrets.")
        client = None

    if client:
        try:
            # --- STEP 1: Create Batch Transcription Job ---
            print(f"📦 Initializing Speech-to-Text Job...")
            job = client.speech_to_text_job.create_job(
                model="saaras:v3",
                mode="transcribe",
                language_code="pa-IN",
                with_diarization=False
            )

            # --- STEP 2: Upload long file ---
            print("📤 Uploading audio to Sarvam cloud storage...")
            job.upload_files(file_paths=[YOUR_AUDIO_FILE])

            print("⚙️ Processing audio script on remote server...")
            job.start()

            # --- STEP 3: Wait for completion ---
            print("⏳ Waiting for batch job to complete...")
            job.wait_until_complete()

            # --- STEP 4: Parse Transcript Payload ---
            with tempfile.TemporaryDirectory() as temp_dir:
                job.download_outputs(output_dir=temp_dir)
                json_files = glob.glob(os.path.join(temp_dir, "*.json"))
                if not json_files:
                    raise Exception("Batch processing failed to yield output files.")

                with open(json_files[0], 'r', encoding='utf-8') as f:
                    result_data = json.load(f)

                punjabi_text = result_data.get("transcript", "")

            print("✅ Full Native Punjabi Transcript extracted successfully.")

            # --- STEP 5: Safe Safe Translation via Helper ---
            print("🔄 Passing transcript to chunked translation module...")
            hindi_text = chunk_and_translate(client, punjabi_text, max_chars=1500)

            # --- STEP 6: Write Final Text Log Files ---
            punjabi_out = "/content/transcript_punjabi.txt"
            hindi_out = "/content/transcript_hindi.txt"

            with open(punjabi_out, "w", encoding="utf-8") as pf:
                pf.write(punjabi_text)

            with open(hindi_out, "w", encoding="utf-8") as hf:
                hf.write(hindi_text)

            print("\n✨ TASK 1 COMPLETION SUCCESSFUL!")
            print(f"💾 File 1 (Gurmukhi) Saved: {punjabi_out}")
            print(f"💾 File 2 (Devanagari) Saved: {hindi_out}")
            print("👉 Click the Refresh button on the sidebar to download both files!")

        except ApiError as e:
            print(f"\n❌ Sarvam API Error ({e.status_code}): {e.body}")
        except Exception as e:
            print(f"\n❌ Pipeline Crash: {str(e)}")

🚀 Starting Task 1 (Chunked Batch Mode)...
📦 Initializing Speech-to-Text Job...
📤 Uploading audio to Sarvam cloud storage...
⚙️ Processing audio script on remote server...
⏳ Waiting for batch job to complete...
✅ Full Native Punjabi Transcript extracted successfully.
🔄 Passing transcript to chunked translation module...
✂️ Text split into 10 smaller segments for safe processing...
   🔄 Processing segment 1/10...
   🔄 Processing segment 2/10...
   🔄 Processing segment 3/10...
   🔄 Processing segment 4/10...
   🔄 Processing segment 5/10...
   ❌ Failed to translate segment 5: {'error': {'message': 'Internal server error', 'code': 'internal_server_error', 'request_id': '20260611_fab09c37-90f7-40c4-8ded-d790f14d557b'}}
   🔄 Processing segment 6/10...
   🔄 Processing segment 7/10...
   🔄 Processing segment 8/10...
   🔄 Processing segment 9/10...
   🔄 Processing segment 10/10...

✨ TASK 1 COMPLETION SUCCESSFUL!
💾 File 1 (Gurmukhi) Saved: /content/transcript_punjabi.txt
💾 File 2 (Devanagari) Sa

In [ ]:
import os
import glob
import tempfile
import time
import json
from google.colab import userdata
from sarvamai import SarvamAI
from sarvamai.core.api_error import ApiError

# =====================================================================
# CONFIGURATION: Set your long audio file name here
# =====================================================================
YOUR_AUDIO_FILE = "/content/AUD-20260402-WA0018.m4a"

# =====================================================================
# HELPER FUNCTION: Safely split text by sentences under character limit
# =====================================================================
def chunk_and_translate(client, text, max_chars=1500):
    """Splits text into safe chunks and translates them one by one"""
    if not text.strip():
        return ""

    # Split text into sentences using common Indian punctuation markers
    sentences = text.replace('।', '.').split('. ')
    chunks = []
    current_chunk = ""

    for sentence in sentences:
        if len(current_chunk) + len(sentence) + 2 < max_chars:
            current_chunk += sentence + "। "
        else:
            chunks.append(current_chunk.strip())
            current_chunk = sentence + "। "
    if current_chunk:
        chunks.append(current_chunk.strip())

    translated_chunks = []

    for idx, chunk in enumerate(chunks):
        if not chunk:
            continue
        try:
            translit_response = client.text.translate(
                input=chunk,
                source_language_code="pa-IN",
                target_language_code="hi-IN",
                model="sarvam-translate:v1"
            )
            translated_chunks.append(translit_response.translated_text)
            time.sleep(0.5) # Small pause to prevent API rate-limiting
        except ApiError as e:
            print(f"   ❌ Failed to translate segment {idx + 1}: {e.body}")

    return " ".join(translated_chunks)

# =====================================================================
# ASYNC BATCH PIPELINE ENGINE
# =====================================================================
print("🚀 Starting Task 1 (Chunked Batch Mode with Diarization)...")

if not os.path.exists(YOUR_AUDIO_FILE):
    print(f"❌ File Not Found: Cannot find '{YOUR_AUDIO_FILE}'")
else:
    try:
        API_KEY = userdata.get('SARVAM_API_KEY')
        client = SarvamAI(api_subscription_key=API_KEY)
    except Exception:
        print("❌ Secret Key Error: 'SARVAM_API_KEY' not found in Colab Secrets.")
        client = None

    if client:
        try:
            # --- STEP 1: Create Batch Transcription Job (With Diarization) ---
            print(f"📦 Initializing Speech-to-Text Job with Diarization...")
            job = client.speech_to_text_job.create_job(
                model="saaras:v3",
                mode="transcribe",
                language_code="pa-IN",
                with_diarization=True  # 👈 CHANGED: Enabled Speaker Identification
            )

            # --- STEP 2: Upload long file ---
            print("📤 Uploading audio to Sarvam cloud storage...")
            job.upload_files(file_paths=[YOUR_AUDIO_FILE])

            print("⚙️ Processing audio script on remote server...")
            job.start()

            # --- STEP 3: Wait for completion ---
            print("⏳ Waiting for batch job to complete...")
            job.wait_until_complete()

            # --- STEP 4: Parse Transcript Payload ---
            with tempfile.TemporaryDirectory() as temp_dir:
                job.download_outputs(output_dir=temp_dir)
                json_files = glob.glob(os.path.join(temp_dir, "*.json"))
                if not json_files:
                    raise Exception("Batch processing failed to yield output files.")

                with open(json_files[0], 'r', encoding='utf-8') as f:
                    result_data = json.load(f)

            # --- STEP 5: Process Diarized Transcript ---
            punjabi_lines = []
            hindi_lines = []

            # When diarization is True, the API usually returns a list of segments
            segments = result_data.get("segments", [])

            if not segments:
                # Fallback if structure differs slightly, check for baseline transcript
                print("⚠️ No speaker segments found. Falling back to standard transcript.")
                punjabi_text = result_data.get("transcript", "")
                punjabi_lines.append(punjabi_text)

                print("🔄 Passing transcript to chunked translation module...")
                hindi_text = chunk_and_translate(client, punjabi_text, max_chars=1500)
                hindi_lines.append(hindi_text)
            else:
                print(f"✅ Extracted {len(segments)} spoken segments. Translating conversations...")

                # We group sequential turns or translate turn-by-turn
                for idx, segment in enumerate(segments):
                    speaker = segment.get("speaker", f"Speaker {segment.get('speaker_id', 'Unknown')}")
                    text = segment.get("transcript", "").strip()

                    if not text:
                        continue

                    # Format Punjabi line
                    punjabi_line = f"{speaker}: {text}"
                    punjabi_lines.append(punjabi_line)

                    # Translate this speaker's chunk
                    print(f"   🔄 Translating segment {idx + 1}/{len(segments)} ({speaker})...")
                    translated_text = chunk_and_translate(client, text, max_chars=1500)

                    # Format Hindi line
                    hindi_line = f"{speaker}: {translated_text}"
                    hindi_lines.append(hindi_line)

            # Join lines with newlines for readable transcripts
            punjabi_final_text = "\n".join(punjabi_lines)
            hindi_final_text = "\n".join(hindi_lines)

            # --- STEP 6: Write Final Text Log Files ---
            punjabi_out = "/content/transcript_punjabi.txt"
            hindi_out = "/content/transcript_hindi.txt"

            with open(punjabi_out, "w", encoding="utf-8") as pf:
                pf.write(punjabi_final_text)

            with open(hindi_out, "w", encoding="utf-8") as hf:
                hf.write(hindi_final_text)

            print("\n✨ TASK 1 COMPLETION SUCCESSFUL!")
            print(f"💾 File 1 (Gurmukhi with Speakers) Saved: {punjabi_out}")
            print(f"💾 File 2 (Devanagari with Speakers) Saved: {hindi_out}")
            print("👉 Click the Refresh button on the sidebar to download both files!")

        except ApiError as e:
            print(f"\n❌ Sarvam API Error ({e.status_code}): {e.body}")
        except Exception as e:
            print(f"\n❌ Pipeline Crash: {str(e)}")

🚀 Starting Task 1 (Chunked Batch Mode with Diarization)...
📦 Initializing Speech-to-Text Job with Diarization...
📤 Uploading audio to Sarvam cloud storage...
⚙️ Processing audio script on remote server...
⏳ Waiting for batch job to complete...
⚠️ No speaker segments found. Falling back to standard transcript.
🔄 Passing transcript to chunked translation module...

✨ TASK 1 COMPLETION SUCCESSFUL!
💾 File 1 (Gurmukhi with Speakers) Saved: /content/transcript_punjabi.txt
💾 File 2 (Devanagari with Speakers) Saved: /content/transcript_hindi.txt
👉 Click the Refresh button on the sidebar to download both files!
